# 🎙️ Kin-AI-Avatar: Ultra-Fast OmniVoice GPU Server

> **💡 TIP:** To run **both Voice (OmniVoice) and Avatar (MuseTalk) in ONE notebook** with a single public URL, use [`Backend/kin_avatar_unified_colab.ipynb`](../kin_avatar_unified_colab.ipynb)!

High-speed, real-time voice cloning server with **`VoiceClonePrompt` caching** and **optimized diffusion steps (`num_step=16`)**.

---
### ⚡ Quickstart:
1. **Runtime > Change runtime type > T4 GPU** (or A100).
2. Run all cells in order (`Shift + Enter`).
3. Copy the **Public URL** into `Backend/.env` as `COLAB_VOICE_URL` (or `COLAB_SERVER_URL`).

### Step 1: Install Dependencies
Install `omnivoice`, FastAPI, and tunnel tools.

In [ ]:
!nvidia-smi
!pip install -q omnivoice soundfile torch torchaudio fastapi uvicorn python-multipart pyngrok pycloudflared WeTextProcessing

### Step 2: Initialize OmniVoice Model

In [ ]:
import torch
from omnivoice import OmniVoice, VoiceClonePrompt
import soundfile as sf

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Loading k2-fsa/OmniVoice on {device}...")

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=device,
    dtype=torch.float16,
    load_asr=True
)

print("✅ OmniVoice Model successfully initialized and cached in VRAM!")

### Step 3: Start Real-Time OmniVoice Server (`num_step=16` for 2x speedup)

In [ ]:
import os
import io
import re
import base64
import json
from pathlib import Path
from typing import Optional, Dict
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import Response, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading

app = FastAPI(title="OmniVoice Colab Server")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

VOICE_DIR = Path("voices")
VOICE_DIR.mkdir(parents=True, exist_ok=True)

cached_prompts: Dict[str, VoiceClonePrompt] = {}
cached_ref_audios: Dict[str, str] = {}

# Preload existing prompts from disk
for pt_file in VOICE_DIR.glob("*.pt"):
    try:
        cached_prompts[pt_file.stem] = VoiceClonePrompt.load(str(pt_file))
        print(f"[Preload] Loaded voice prompt: {pt_file.name}")
    except Exception as e:
        print(f"[Preload error] {e}")

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "engine": "OmniVoice (k2-fsa/OmniVoice)",
        "device": "cuda:0" if torch.cuda.is_available() else "cpu",
        "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None",
        "cached_prompts": list(cached_prompts.keys())
    }

@app.post("/register_voice")
async def register_voice(name: str = Form("default"), ref_text: Optional[str] = Form(None), file: UploadFile = File(...)):
    clean_name = re.sub(r'[^a-zA-Z0-9_-]', '_', name.strip()) or "default"
    ext = Path(file.filename or "sample.wav").suffix or ".wav"
    audio_path = VOICE_DIR / f"{clean_name}{ext}"
    contents = await file.read()
    with open(audio_path, "wb") as f:
        f.write(contents)
    cached_ref_audios[clean_name] = str(audio_path)
    
    prompt_path = VOICE_DIR / f"{clean_name}.pt"
    try:
        prompt = model.create_voice_clone_prompt(ref_audio=str(audio_path), ref_text=ref_text)
        prompt.save(str(prompt_path))
        cached_prompts[clean_name] = prompt
        return {"status": "success", "speaker_name": clean_name, "cached": True, "message": f"Voice prompt '{clean_name}' created and cached."}
    except Exception as e:
        cached_prompts[clean_name] = str(audio_path)
        return {"status": "partial_success", "speaker_name": clean_name, "message": f"Saved reference audio fallback for '{clean_name}'."}

def get_target(speaker_name: str):
    if speaker_name in cached_prompts:
        return cached_prompts[speaker_name]
    pt_path = VOICE_DIR / f"{speaker_name}.pt"
    if pt_path.exists():
        prompt = VoiceClonePrompt.load(str(pt_path))
        cached_prompts[speaker_name] = prompt
        return prompt
    for ext in [".wav", ".mp3", ".flac"]:
        p = VOICE_DIR / f"{speaker_name}{ext}"
        if p.exists():
            return str(p)
    if cached_prompts:
        return next(iter(cached_prompts.values()))
    raise HTTPException(status_code=400, detail="No voice registered yet. Call /register_voice first.")

def synth_tensor(text: str, target, num_step: int = 16):
    kw = {"text": text, "normalize_text": False, "num_step": num_step}
    if isinstance(target, VoiceClonePrompt):
        kw["voice_clone_prompt"] = target
    else:
        kw["ref_audio"] = str(target)
    with torch.inference_mode():
        try:
            out = model.generate(**kw)
        except TypeError:
            kw.pop("num_step", None)
            kw.pop("normalize_text", None)
            out = model.generate(**kw)
    arr = out[0] if isinstance(out, (list, tuple)) else out
    return arr.cpu().numpy() if hasattr(arr, "cpu") else arr

def to_wav(arr, rate=24000):
    b = io.BytesIO()
    sf.write(b, arr, rate, format="WAV")
    return b.getvalue()

class SynthReq(BaseModel):
    text: str
    speaker_name: str = "default"
    num_step: int = 16

@app.post("/synthesize")
def synthesize(req: SynthReq):
    target = get_target(req.speaker_name)
    arr = synth_tensor(req.text.strip(), target, num_step=req.num_step)
    return Response(content=to_wav(arr), media_type="audio/wav")

@app.post("/synthesize_stream")
def synthesize_stream(req: SynthReq):
    target = get_target(req.speaker_name)
    tokens = re.split(r'([.!?;:\n]+)', req.text.strip())
    sentences = []
    for i in range(0, len(tokens) - 1, 2):
        p = tokens[i].strip() + (tokens[i+1].strip() if i+1 < len(tokens) else "")
        if p: sentences.append(p)
    if len(tokens) % 2 == 1 and tokens[-1].strip():
        sentences.append(tokens[-1].strip())
    if not sentences: sentences = [req.text.strip()]

    clauses = []
    for idx, s in enumerate(sentences):
        words = s.split()
        if idx == 0 and len(words) > 7 and (',' in s or '—' in s):
            parts = re.split(r'([,;—]+)', s)
            fc = parts[0].strip() + (parts[1].strip() if len(parts) > 1 else "")
            rst = "".join(parts[2:]).strip()
            if fc and rst:
                clauses.append(fc)
                clauses.append(rst)
                continue
        clauses.append(s)

    def gen():
        for idx, cl in enumerate(clauses):
            try:
                arr = synth_tensor(cl, target, num_step=req.num_step)
                b64 = base64.b64encode(to_wav(arr)).decode("utf-8")
                yield json.dumps({"chunk_index": idx, "total": len(clauses), "text": cl, "audio_base64": b64, "is_last": idx == len(clauses) - 1}) + "\n"
            except Exception as ex:
                yield json.dumps({"chunk_index": idx, "error": str(ex), "is_last": idx == len(clauses) - 1}) + "\n"

    return StreamingResponse(gen(), media_type="application/x-ndjson")

def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
print("✅ Real-Time OmniVoice Server running on port 8000!")

### Step 4: Expose Public URL

In [ ]:
from pycloudflared import try_cloudflare
try:
    tunnel = try_cloudflare(port=8000)
    public_url = tunnel.tunnel_url
    print("=" * 70)
    print(f"🚀 PUBLIC COLAB URL: {public_url}")
    print("=" * 70)
    print(f"COLAB_VOICE_URL = {public_url}")
    print("=" * 70)
except Exception as e:
    print(f"Cloudflared fallback: {e}")
    from pyngrok import ngrok
    tunnel = ngrok.connect(8000)
    public_url = tunnel.public_url
    print(f"🚀 PUBLIC COLAB URL: {public_url}")